# Customize MOM output

## Adding MOM6 Diagnostics

MOM6 diagnostics are controlled through a file called `diag_table`. 

During case setup, CESM generates a template file named:

```bash
Buildconf/momconf/diag_table.unresolved
```

To customize the diagnostics that MOM6 outputs, copy this file into your SourceMods directory:

```bash
cp Buildconf/momconf/diag_table.unresolved SourceMods/src.mom/diag_table
```

You can then edit `SourceMods/src.mom/diag_table` to add, remove, or modify diagnostic output fields.

**Note** *CESM regenerates files in `Buildconf` during case setup. Therefore, **user modifications should be made in `SourceMods/src.mom/diag_table`**, which will override the default version and be preserved during subsequent builds.*

The `diag_table` specifies:

* Which variables to output
* How frequently they are written
* Whether they are snapshots or time averages
* Which output files they are written to

A diagnostic entry has the form:

```text
"module_name", "field_name", "output_name", "file_name", "time_sampling", "reduction_method", "regional_section", packing
```

For example:

```text
"ocean_model", "thetao", "thetao", "ocean_month", "all", "mean", "none", 2
```

writes monthly mean potential temperature (`thetao`).

### Finding Available Diagnostics

The set of available diagnostics depends on the MOM6 configuration being used. After running the model, MOM6 writes a file named:

```text
available_diags.000000
```

in the run directory. This file contains a complete list of diagnostics available for your configuration.

To view the available diagnostics:

```bash
cd $RUNDIR
less available_diags.000000
```

or search for a specific variable:

```bash
grep temp available_diags.000000
```

Before adding new diagnostics to the `diag_table`, it is often helpful to inspect `available_diags.000000` to determine the exact diagnostic names recognized by MOM6.

### Changing the Output Frequency

The output frequency is controlled in the **file definition** section of the `diag_table`.

For example:

```text
"ocean_month", 1, "months", 1, "days", "time"
```

creates a file named `ocean_month.nc` with output written every month.

The general format is:

```text
"file_name", output_freq, "output_freq_units", 1, "time_axis_units", "time_axis_name"
```

Common examples include:

```text
"ocean_hourly", 1, "hours", 1, "days", "time"
"ocean_daily",  1, "days",  1, "days", "time"
"ocean_month",  1, "months",1, "days", "time"
"ocean_annual", 12, "months",1, "days", "time"
```

To change monthly output to daily output, modify the file definition:

```text
"ocean_daily", 1, "days", 1, "days", "time"
```

and update the corresponding diagnostic entries to write to the new file:

```text
"ocean_model", "temp", "temp", "ocean_daily", "all", "mean", "none", 2
```

### Snapshots versus Time Averages

The output frequency determines **when data are written**, while the reduction method determines **how data are accumulated between writes**.

Common reduction methods are:

```text
"none"    # instantaneous snapshot
"mean"    # time average
"min"     # minimum value
"max"     # maximum value
```

For example:

```text
"ocean_model", "SSH", "SSH",
"ocean_daily", "all", "none", "none", 2
```

writes a daily snapshot of sea surface height, while

```text
"ocean_model", "temp", "temp",
"ocean_daily", "all", "mean", "none", 2
```

writes a daily mean temperature.

**Note**: *Increasing the output frequency or adding a large number of diagnostics can **significantly increase output volume and reduce model throughput**. Start with a small number of diagnostics and increase the frequency only when needed.*

For additional details about the `diag_table` format and MOM6 diagnostics, see the MOM6 documentation.
https://mom6.readthedocs.io/en/main/api/generated/pages/Diagnostics.html

